# 05 - Feature Engineering with NumPy and Pandas

This learning notebook covers **Part 2, Task 2**. It converts the cleaned record-level data into a one-row-per-hour modelling table and creates reusable time, weather, numerical and target features.

## Learning objectives

1. Consolidate repeated weather observations without duplicating traffic targets.
2. Create hour, day-of-week and weekend features.
3. Create cyclical encodings with NumPy.
4. Encode multiple weather conditions as hourly binary flags.
5. Create precipitation, low-visibility and severe-weather indicators.
6. Scale at least two continuous variables.
7. Create a documented quartile-based congestion target.
8. Log input/output shapes and DEBUG-level intermediate thresholds.

> Full-dataset scaling in this notebook satisfies the feature-engineering exercise and supports descriptive tools. During supervised model evaluation, scaling parameters must be fitted on training data only to prevent leakage.

## 1. Import packages and configure DEBUG learning logs

In [1]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

LOGGER = logging.getLogger("capstone.learning.feature_engineering")
LOGGER.handlers.clear()
LOGGER.setLevel(logging.DEBUG)
LOGGER.propagate = False
handler = logging.StreamHandler()
handler.setLevel(logging.DEBUG)
handler.setFormatter(
    logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s")
)
LOGGER.addHandler(handler)
LOGGER.info("Feature-engineering environment ready: Pandas %s, NumPy %s", pd.__version__, np.__version__)

2026-11-14 23:33:30,157 | INFO | capstone.learning.feature_engineering | Feature-engineering environment ready: Pandas 3.1.1, NumPy 2.4.6


## 2. Load the cleaned record-level dataset

The cleaned dataset contains 48,187 records but only 40,575 timestamps because one hour can contain several weather descriptions.

In [2]:
CLEANED_RELATIVE_PATH = Path("part2_python/data/processed/traffic_cleaned.csv")

def find_project_root(start: Path, relative_path: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / relative_path).is_file():
            return candidate
    raise FileNotFoundError(f"Could not find {relative_path}. Run pipeline.py first.")

PROJECT_ROOT = find_project_root(Path.cwd(), CLEANED_RELATIVE_PATH)
CLEANED_PATH = PROJECT_ROOT / CLEANED_RELATIVE_PATH
cleaned = pd.read_csv(
    CLEANED_PATH,
    parse_dates=["date_time"],
    keep_default_na=False,
)
LOGGER.info("Dataset shape before feature engineering: %s", cleaned.shape)
display(cleaned.head(3))

2026-11-14 23:33:30,186 | INFO | capstone.learning.feature_engineering | Dataset shape before feature engineering: (48187, 9)


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,None,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,None,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,None,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767


## 3. Verify target consistency and consolidate to one hourly row

Traffic volume must not conflict within a timestamp. Continuous weather measurements use median or maximum values, while a non-`None` holiday label is retained when present.

In [3]:
traffic_conflicts = int(
    cleaned.groupby("date_time")["traffic_volume"].nunique().gt(1).sum()
)
if traffic_conflicts:
    raise ValueError(f"Found {traffic_conflicts} timestamps with conflicting traffic targets")
LOGGER.info("Traffic target consistency passed for all timestamps")

def select_holiday(values: pd.Series) -> str:
    labelled = sorted({value for value in values if value != "None"})
    return labelled[0] if labelled else "None"

hourly = (
    cleaned.groupby("date_time", as_index=False)
    .agg(
        holiday=("holiday", select_holiday),
        temp=("temp", "median"),
        rain_1h=("rain_1h", "max"),
        snow_1h=("snow_1h", "max"),
        clouds_all=("clouds_all", "median"),
        traffic_volume=("traffic_volume", "first"),
    )
)
LOGGER.info("Consolidated %d records into %d unique hourly rows", len(cleaned), len(hourly))

2026-11-14 23:33:30,196 | INFO | capstone.learning.feature_engineering | Traffic target consistency passed for all timestamps


2026-11-14 23:33:30,366 | INFO | capstone.learning.feature_engineering | Consolidated 48187 records into 40575 unique hourly rows


## 4. Encode hourly weather conditions

One timestamp may contain more than one high-level weather condition. A multi-hot representation preserves every observed condition instead of forcing one arbitrary category. Each `weather_*` feature is therefore 1 when that condition occurred during the hour.

In [4]:
weather_flags = pd.crosstab(cleaned["date_time"], cleaned["weather_main"])
weather_flags = weather_flags.clip(upper=1).astype("int8")
weather_flags.columns = [f"weather_{column}" for column in weather_flags.columns]
weather_flags = weather_flags.reset_index()

description_severity = (
    cleaned.assign(
        _description_severe=cleaned["weather_description"].str.contains(
            r"heavy|thunderstorm|squall|freezing",
            case=False,
            regex=True,
            na=False,
        ).astype("int8")
    )
    .groupby("date_time", as_index=False)["_description_severe"]
    .max()
)

features = hourly.merge(weather_flags, on="date_time", how="left", validate="one_to_one")
features = features.merge(
    description_severity,
    on="date_time",
    how="left",
    validate="one_to_one",
)
weather_feature_columns = sorted(
    column for column in features.columns if column.startswith("weather_")
)
LOGGER.info("Created %d multi-hot weather features", len(weather_feature_columns))
LOGGER.debug("Weather feature columns: %s", weather_feature_columns)

2026-11-14 23:33:30,516 | INFO | capstone.learning.feature_engineering | Created 11 multi-hot weather features


2026-11-14 23:33:30,516 | DEBUG | capstone.learning.feature_engineering | Weather feature columns: ['weather_Clear', 'weather_Clouds', 'weather_Drizzle', 'weather_Fog', 'weather_Haze', 'weather_Mist', 'weather_Rain', 'weather_Smoke', 'weather_Snow', 'weather_Squall', 'weather_Thunderstorm']


## 5. Create time, holiday and derived weather indicators

Cyclical encodings place hour 23 close to hour 0 and Sunday close to Monday in two-dimensional sine/cosine space. This avoids treating the endpoints as far apart.

In [5]:
features["hour"] = features["date_time"].dt.hour.astype("int8")
features["day_of_week"] = features["date_time"].dt.day_of_week.astype("int8")
features["year"] = features["date_time"].dt.year.astype("int16")
features["month"] = features["date_time"].dt.month.astype("int8")
features["is_weekend"] = features["day_of_week"].isin([5, 6]).astype("int8")
features["is_holiday"] = features["holiday"].ne("None").astype("int8")

features["hour_sin"] = np.sin(2 * np.pi * features["hour"] / 24)
features["hour_cos"] = np.cos(2 * np.pi * features["hour"] / 24)
features["day_sin"] = np.sin(2 * np.pi * features["day_of_week"] / 7)
features["day_cos"] = np.cos(2 * np.pi * features["day_of_week"] / 7)

def any_weather(columns: list[str]) -> pd.Series:
    existing = [column for column in columns if column in features.columns]
    if not existing:
        return pd.Series(0, index=features.index, dtype="int8")
    return features[existing].max(axis=1).astype("int8")

features["is_precipitation"] = (
    any_weather(
        [
            "weather_Rain", "weather_Drizzle", "weather_Snow",
            "weather_Thunderstorm", "weather_Squall",
        ]
    ).eq(1)
    | features["rain_1h"].gt(0)
    | features["snow_1h"].gt(0)
).astype("int8")
features["is_low_visibility"] = any_weather(
    ["weather_Mist", "weather_Fog", "weather_Haze", "weather_Smoke"]
)
features["is_severe_weather"] = (
    any_weather(["weather_Thunderstorm", "weather_Squall"]).eq(1)
    | features["_description_severe"].eq(1)
).astype("int8")
features = features.drop(columns=["_description_severe"])

LOGGER.info("Created time, cyclical, holiday and derived weather indicators")

2026-11-14 23:33:30,527 | INFO | capstone.learning.feature_engineering | Created time, cyclical, holiday and derived weather indicators


## 6. Scale and transform continuous variables

Temperature and cloud coverage receive z-score versions. Rain and snow receive `log1p` transforms because most values are zero and a few are much larger. Intermediate means and standard deviations are DEBUG-only values.

In [6]:
scaling_parameters = {}
for source_column, output_column in [
    ("temp", "temp_zscore"),
    ("clouds_all", "clouds_all_zscore"),
]:
    mean_value = float(np.mean(features[source_column]))
    standard_deviation = float(np.std(features[source_column], ddof=0))
    if standard_deviation == 0:
        raise ValueError(f"Cannot scale constant feature {source_column}")
    features[output_column] = (features[source_column] - mean_value) / standard_deviation
    scaling_parameters[source_column] = {
        "mean": mean_value,
        "population_standard_deviation": standard_deviation,
    }
    LOGGER.debug(
        "Scaling parameters for %s: mean=%.6f std=%.6f",
        source_column,
        mean_value,
        standard_deviation,
    )

features["rain_log1p"] = np.log1p(features["rain_1h"])
features["snow_log1p"] = np.log1p(features["snow_1h"])
LOGGER.info("Created scaled versions of two continuous variables and two log transforms")

2026-11-14 23:33:30,531 | DEBUG | capstone.learning.feature_engineering | Scaling parameters for temp: mean=281.380010 std=13.096263


2026-11-14 23:33:30,532 | DEBUG | capstone.learning.feature_engineering | Scaling parameters for clouds_all: mean=44.201121 std=38.680787


2026-11-14 23:33:30,532 | INFO | capstone.learning.feature_engineering | Created scaled versions of two continuous variables and two log transforms


## 7. Create the data-driven congestion target

The target uses the hourly traffic-volume quartiles required for later Part 3 reuse:

- Low: value at or below Q1
- Medium: above Q1 and at or below the median
- High: above the median and at or below Q3
- Severe: above Q3

Thresholds are logged at DEBUG level because they are intermediate calculations.

In [7]:
q1, q2, q3 = features["traffic_volume"].quantile([0.25, 0.50, 0.75]).tolist()
LOGGER.debug("Congestion quartiles: q1=%.4f q2=%.4f q3=%.4f", q1, q2, q3)

def bucket_congestion(value: float) -> str:
    if value <= q1:
        return "Low"
    if value <= q2:
        return "Medium"
    if value <= q3:
        return "High"
    return "Severe"

features["congestion_category"] = pd.Categorical(
    features["traffic_volume"].map(bucket_congestion),
    categories=["Low", "Medium", "High", "Severe"],
    ordered=True,
)
target_counts = features["congestion_category"].value_counts(sort=False)
LOGGER.info("Created quartile-based congestion target with four ordered categories")
display(target_counts.rename("hourly_rows").to_frame())

2026-11-14 23:33:30,536 | DEBUG | capstone.learning.feature_engineering | Congestion quartiles: q1=1248.5000 q2=3427.0000 q3=4952.0000


2026-11-14 23:33:30,542 | INFO | capstone.learning.feature_engineering | Created quartile-based congestion target with four ordered categories


,hourly_rows
congestion_category,
Low,10144
Medium,10144
High,10145
Severe,10142


## 8. Validate and inspect the feature table

In [8]:
features = features.sort_values("date_time").reset_index(drop=True)
LOGGER.info("Dataset shape after feature engineering: %s", features.shape)

feature_summary = {
    "input_record_rows": len(cleaned),
    "output_hourly_rows": len(features),
    "output_columns": len(features.columns),
    "weather_feature_count": len(weather_feature_columns),
    "missing_values": int(features.isna().sum().sum()),
    "duplicate_timestamps": int(features["date_time"].duplicated().sum()),
    "q1": q1,
    "q2": q2,
    "q3": q3,
}
display(pd.Series(feature_summary, name="value").to_frame())
display(features.head(3))

2026-11-14 23:33:30,547 | INFO | capstone.learning.feature_engineering | Dataset shape after feature engineering: (40575, 36)


,value
input_record_rows,48187.0
output_hourly_rows,40575.0
output_columns,36.0
weather_feature_count,11.0
missing_values,0.0
duplicate_timestamps,0.0
q1,1248.5
q2,3427.0
q3,4952.0


,date_time,holiday,temp,rain_1h,snow_1h,clouds_all,traffic_volume,weather_Clear,weather_Clouds,weather_Drizzle,...,day_sin,day_cos,is_precipitation,is_low_visibility,is_severe_weather,temp_zscore,clouds_all_zscore,rain_log1p,snow_log1p,congestion_category
0,2012-10-02 09:00:00,None,288.28,0.0,0.0,40.0,5545,0,1,0,...,0.781831,0.62349,0,0,0,0.526867,-0.108610,0.0,0.0,Severe
1,2012-10-02 10:00:00,None,289.36,0.0,0.0,75.0,4516,0,1,0,...,0.781831,0.62349,0,0,0,0.609333,0.796232,0.0,0.0,High
2,2012-10-02 11:00:00,None,289.58,0.0,0.0,90.0,4767,0,1,0,...,0.781831,0.62349,0,0,0,0.626132,1.184021,0.0,0.0,High


## 9. Run automated checks

In [9]:
assert cleaned.shape == (48_187, 9)
assert traffic_conflicts == 0
assert features.shape == (40_575, 36)
assert len(weather_feature_columns) == 11
assert features["date_time"].is_monotonic_increasing
assert not features["date_time"].duplicated().any()
assert not features.isna().any().any()
assert (q1, q2, q3) == (1248.5, 3427.0, 4952.0)
assert target_counts.to_dict() == {
    "Low": 10_144,
    "Medium": 10_144,
    "High": 10_145,
    "Severe": 10_142,
}
assert np.isclose(features["temp_zscore"].mean(), 0.0, atol=1e-12)
assert np.isclose(features["temp_zscore"].std(ddof=0), 1.0, atol=1e-12)
assert np.isclose(features["clouds_all_zscore"].mean(), 0.0, atol=1e-12)
assert np.isclose(features["clouds_all_zscore"].std(ddof=0), 1.0, atol=1e-12)
assert features[weather_feature_columns].isin([0, 1]).all().all()
assert features[["hour_sin", "hour_cos", "day_sin", "day_cos"]].apply(
    lambda series: series.between(-1, 1).all()
).all()

print("PASS - Hourly feature engineering matched all expected results.")

PASS - Hourly feature engineering matched all expected results.


## 10. Interpretation checkpoint

Before creating `feature_engineering.py`, make sure you can explain:

1. Why the modelling table has one row per timestamp.
2. Why weather uses multi-hot rather than one arbitrary category.
3. Why sine and cosine are both required for a cyclical feature.
4. What z-score scaling changes and what it preserves.
5. Why scaling parameters must later be fitted on training data only.
6. How the quartile thresholds produce Low, Medium, High and Severe targets.
7. Why target-derived fields must never be used as predictors for that target.

### Stop point

Run all cells and confirm the final code cell reports `PASS`. Save the notebook. We will then promote this logic into `feature_engineering.py`, generate the ML-ready hourly CSV, add tests and extend the logging trail.